# Repair baseline: zero-shot edit programs from Qwen3.5-9B

Is training needed at all? The answer generator itself (Qwen3.5-9B, 4-bit on llama.cpp, thinking off) writes edit programs zero-shot, its replies constrained to the program schema, on the same 500 tune-track items as the trained 2B repairer. Expect about 1 GPU hour.

**On Kaggle:** Create → New Notebook → File → Import Notebook → this file. Settings: *Accelerator* **GPU T4 x2**, *Internet* **on**.
For long jobs use **Save Version → Save & Run All (Commit)**: it runs in the background for up to 12 hours and keeps everything in `/kaggle/working/outputs`.
To continue in a later notebook, add this notebook's output as input (*Add Input → Your Work*).

**Afterwards:** download `outputs/*/results.json` and `report.md` files into `runs/` in the repository, and log the session's GPU hours in `logs/gpu-hours.csv`. Checkpoints are saved regularly; re-running the same cell resumes from the latest one.

In [ ]:
# Setup: clone FaithGuard, install it with the pinned GPU packages, fetch the data.
import shutil, socket
try:
    socket.gethostbyname("pypi.org")
except OSError:
    raise RuntimeError("No internet in this session. Kaggle: verify your phone number (Settings), then switch Internet on in the right-hand panel.")
if not shutil.which("nvidia-smi"):
    raise RuntimeError("No GPU in this session. Kaggle: verify your phone number (Settings), then set Accelerator to GPU T4 x2. Colab: Runtime > Change runtime type > T4 GPU.")
import os, subprocess, sys
WORK = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
os.chdir(WORK)
if not os.path.exists('faithguard'):
    subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/SharafThawfeek/FaithGurd-new.git', 'faithguard'])
os.chdir('faithguard')
REQUIREMENTS = ['transformers==5.19.0', 'peft==0.21.2', 'accelerate==1.15.0', 'bitsandbytes==0.50.2', 'huggingface_hub==1.33.0', 'datasets==5.1.0', 'lettucedetect==0.2.3']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *REQUIREMENTS])
# Kaggle's image ships torchao 0.10, which PEFT 0.21 rejects when it adds LoRA to fp16 weights; nothing here uses it
subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])
# this kernel started before the install, and the clone's own folder would shadow the package: import from src/
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
OUT = os.path.join(WORK, 'outputs'); os.makedirs(OUT, exist_ok=True)
!faithguard data download all
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Start llama.cpp with Qwen3.5-9B on one T4 (downloads 5.4 GB the first time)

In [ ]:
from faithguard.generation import llama
SERVER = llama.prepare(os.path.join(WORK, 'llama_work'))
server = llama.LlamaServer(SERVER, llama.model_path('qwen'), port=8080, gpu='0', log=f'{OUT}/server-qwen.log')
print('ready in', round(server.wait_ready()), 's')

## Zero-shot programs on the tune track: free replies (as the 2B zero-shot run), then schema-constrained (D-083)

In [ ]:
!python -m faithguard.train.repair_eval --server http://127.0.0.1:8080 --unconstrained --out {OUT}/eval-zero-shot-9b-free
!python -m faithguard.train.repair_eval --server http://127.0.0.1:8080 --out {OUT}/eval-zero-shot-9b

## Stop the server

In [ ]:
server.stop()

## Results

In [ ]:
import glob, json
for path in sorted(glob.glob(OUT + '/**/results.json', recursive=True)):
    print(path)
    print(json.dumps({k: v for k, v in json.load(open(path)).items() if k in ('summary', 'channel_a', 'channel_b', 'auroc', 'false_alarm_clean', 'wrong_context_recall')}, indent=1)[:1500])